# BIOT 6900 · Module 7 · Assignment Starter (Neo4j)

### *Build a knowledge graph for YOUR Part I disease*

This is the starter for the graded assignment. The database plumbing, the Cypher
queries, and the graph algorithms are **already written and working** — your job
is to supply the **data for your own disease** (the cells marked **`# TODO`**) and
then **interpret** what the graph tells you.

It ships with a tiny placeholder example so it runs top to bottom as a skeleton.
Replace the example with your disease, section by section, re-running as you go.

**Before you start:** create a free Neo4j AuraDB instance and have your
`targets_ad.csv` from Module 2 ready. See the Module 7 lab guide for setup.
Run with **Kernel → Restart & Run All**.


## 0 · Install and connect

Run this, then paste your Neo4j connection details into the next cell. On AuraDB
Free the **username is your instance ID**, not `neo4j`.

In [ ]:
# If the kernel restarts after installing, run the cells again from the top.
%pip install neo4j networkx matplotlib pandas requests

In [ ]:
from neo4j import GraphDatabase

NEO4J_URI  = "neo4j+s://YOUR-INSTANCE.databases.neo4j.io"   # from your credentials file
NEO4J_USER = "YOUR-INSTANCE-ID"      # NEO4J_USERNAME from the file (not "neo4j")
NEO4J_PASSWORD = "YOUR-PASSWORD"     # keep this out of anything you commit

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def run_query(cypher, **params):
    with driver.session() as session:
        return [r.data() for r in session.run(cypher, **params)]

try:
    driver.verify_connectivity(); print("Connected to Neo4j.")
except Exception as e:
    print("Not connected yet — check URI/username/password. Details:", e)

## 1 · Your data  (★ this is the part you edit)

Replace every value marked **`# TODO`** with your own disease. Read your genes
from `targets_ad.csv`, then fill in the phenotypes, pathways, drugs, variants, and
**at least one related disease that shares a phenotype** (that is what powers the
repurposing query in §4).

In [ ]:
import pandas as pd
import networkx as nx
import os

# ===========================================================================
# YOUR DISEASE  —  replace every value marked TODO with your own Part I disease.
# The example values let the notebook run as a skeleton; swap them for yours.
# ===========================================================================
DISEASE = "Example disease"          # TODO: your Part I disease, e.g. "Parkinson's disease"

# --- Genes: read your targets_ad.csv from Module 2, or edit the example list ---
def load_my_genes(path="data/targets_ad.csv"):
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"loaded {len(df)} genes from {path}")
        # TODO: make sure your columns are named:
        #       gene, ensembl, gwas_p, rna_lfc, prot_lfc, druggability
        return [(r.get("gene"), r.get("ensembl", ""), r.get("gwas_p"),
                 r.get("rna_lfc"), r.get("prot_lfc"), r.get("druggability"))
                for _, r in df.iterrows()]
    print(f"(no file at {path}) — using the EXAMPLE genes; replace them with yours")
    return [
        # symbol,   ensembl,            gwas_p, rna_lfc, prot_lfc, druggability
        ("GENE1",   "ENSG00000000001", 1e-20, 0.9, 0.8, 0.40),    # TODO
        ("GENE2",   "ENSG00000000002", 1e-12, 0.5, 0.4, 0.60),    # TODO
        ("DRUGTGT1","ENSG00000000003", None,  0.1, 0.1, 0.90),    # TODO: a drug-target gene
    ]
GENES = load_my_genes()

# --- The other layers: edit ALL of these for your disease ---
PHENOTYPES = ["Symptom A", "Symptom B"]                 # TODO: HPO phenotype terms
PATHWAYS   = ["Pathway 1", "Pathway 2"]                 # TODO: Reactome pathways
GENE_IN_PATHWAY = {                                     # TODO: which gene in which pathway
    "Pathway 1": ["GENE1", "GENE2"],
    "Pathway 2": ["DRUGTGT1"],
}
DRUGS    = [("Drug A", "DRUGTGT1", "approved")]         # TODO: drug -> target gene, status
VARIANTS = [("rsEXAMPLE", "GENE1")]                     # TODO: variant -> gene (from Module 5)

# A related disease that SHARES a phenotype — this is what powers the repurposing query.
RELATED = {
    "name":   "Related disease",                                            # TODO
    "shares": ["Symptom A"],                                                # TODO: shared phenotype(s)
    "genes":  [("RELGENE", "ENSG00000000009", 1e-11, 0.4, 0.3, 0.55)],     # TODO
    "drugs":  [("Related drug (investigational)", "RELGENE", "investigational")],  # TODO
}

# OPTIONAL patient layer — leave [] to skip. (id, age, genotype, prs_percentile, status)
PATIENTS = []   # TODO (optional): e.g. ("P01", 72, "e3/e4", 0.88, "case")

# ---------------------------------------------------------------------------
# Build node/edge collections from the lists above (you shouldn't need to edit).
# ---------------------------------------------------------------------------
NODES, EDGES = {}, []
def add_node(n, k, **p): NODES[n] = dict(kind=k, **p)
def add_edge(s, r, d, **p): EDGES.append(dict(src=s, rel=r, dst=d, **p))

add_node(DISEASE, "disease")
for sym, ens, gp, rna, prot, dg in GENES:
    add_node(sym, "gene", ensembl=ens, gwas_p=gp, rna_lfc=rna, prot_lfc=prot, druggability=dg)
    add_edge(sym, "ASSOCIATED_WITH", DISEASE, pvalue=gp, source=("GWAS" if gp else "pharmacology"))
for ph in PHENOTYPES:
    add_node(ph, "phenotype"); add_edge(DISEASE, "HAS_PHENOTYPE", ph)
for pw in PATHWAYS:
    add_node(pw, "pathway")
for pw, gl in GENE_IN_PATHWAY.items():
    for g in gl:
        if g in NODES: add_edge(g, "PARTICIPATES_IN", pw)
for name, tgt, status in DRUGS:
    add_node(name, "drug", status=status); add_edge(name, "TARGETS", tgt)
for v, g in VARIANTS:
    add_node(v, "variant"); add_edge(v, "OCCURS_IN", g)
# related disease
_r = RELATED
add_node(_r["name"], "disease")
for ph in _r["shares"]:
    if ph not in NODES: add_node(ph, "phenotype")
    add_edge(_r["name"], "HAS_PHENOTYPE", ph)
for sym, ens, gp, rna, prot, dg in _r["genes"]:
    add_node(sym, "gene", ensembl=ens, gwas_p=gp, rna_lfc=rna, prot_lfc=prot, druggability=dg)
    add_edge(sym, "ASSOCIATED_WITH", _r["name"], pvalue=gp, source="GWAS")
for name, tgt, status in _r["drugs"]:
    add_node(name, "drug", status=status); add_edge(name, "TARGETS", tgt)
for pid, age, geno, prs, status in PATIENTS:
    add_node(pid, "patient", age=age, apoe_genotype=geno, prs_percentile=prs, status=status)

print("nodes:", len(NODES), " edges:", len(EDGES))
print("disease:", DISEASE)


## 2 · Load into Neo4j

Provided — no edits needed. Creates a uniqueness constraint, then loads your
nodes and relationships with batched `UNWIND` + `MERGE`.

In [ ]:
run_query("CREATE CONSTRAINT entity_name IF NOT EXISTS "
          "FOR (n:Entity) REQUIRE n.name IS UNIQUE")

KIND_LABEL = {"gene":"Gene","disease":"Disease","phenotype":"Phenotype",
              "pathway":"Pathway","drug":"Drug","variant":"Variant","patient":"Patient"}

for kind, label in KIND_LABEL.items():
    rows = [dict(name=n, **{k: v for k, v in d.items() if k != "kind"})
            for n, d in NODES.items() if d["kind"] == kind]
    if rows:
        run_query(f"UNWIND $rows AS row MERGE (e:{label}:Entity {{name: row.name}}) SET e += row",
                  rows=rows)

from collections import defaultdict
by_rel = defaultdict(list)
for e in EDGES:
    by_rel[e["rel"]].append({"src": e["src"], "dst": e["dst"],
                             "props": {k: v for k, v in e.items() if k not in ("src","dst","rel")}})
for rel, rows in by_rel.items():
    run_query(f"""UNWIND $rows AS row
                  MATCH (a:Entity {{name: row.src}})
                  MATCH (b:Entity {{name: row.dst}})
                  MERGE (a)-[r:{rel}]->(b)
                  SET r += row.props""", rows=rows)

print("node counts:", run_query("MATCH (n) RETURN labels(n)[0] AS label, count(*) AS n ORDER BY n DESC"))
print("relationships:", run_query("MATCH ()-[r]->() RETURN count(r) AS n")[0]["n"])

## 3 · A quick look

Provided. Open your instance in the Aura console and run
`MATCH (n) RETURN n LIMIT 100` to see the graph you just loaded, or use §7 to draw it.

## 4 · Query it — the opening question (Cypher)

Provided. These answer the opening question for whatever `DISEASE` you set in §1.

> **TODO (report):** after running, note which drugs come back, and for the
> repurposing query, which related disease and shared symptom produced each hit.

In [ ]:
# 2 hops: drugs acting on genes associated with your disease
run_query("""
    MATCH (d:Drug)-[:TARGETS]->(g:Gene)-[:ASSOCIATED_WITH]->(dis:Disease)
    WHERE dis.name = $disease
    RETURN d.name AS drug, g.name AS gene, d.status AS status
    ORDER BY drug
""", disease=DISEASE)

In [ ]:
# 5 hops: drugs on genes of diseases that SHARE A SYMPTOM with your disease
run_query("""
    MATCH (d:Drug)-[:TARGETS]->(g:Gene)-[:ASSOCIATED_WITH]->(other:Disease)
          -[:HAS_PHENOTYPE]->(p:Phenotype)<-[:HAS_PHENOTYPE]-(ad:Disease)
    WHERE ad.name = $disease AND other <> ad
    RETURN d.name AS repurposing_candidate, g.name AS gene,
           other.name AS via_disease, collect(DISTINCT p.name) AS shared_symptoms
    ORDER BY repurposing_candidate
""", disease=DISEASE)

## 5 · Biomarker prioritization (centrality)

Provided. PageRank fused with your gene evidence.

> **TODO (report):** which genes rank highest? Does that match the biology you
> expect for your disease? Flag any non-obvious hit.

In [ ]:
# PageRank in NetworkX, fused with your gene evidence (runs on any Neo4j tier).
G = nx.DiGraph()
for name, d in NODES.items():
    G.add_node(name, **d)
for e in EDGES:
    G.add_edge(e["src"], e["dst"], **{k: v for k, v in e.items() if k not in ("src", "dst")})

pr = nx.pagerank(G)
rows = [{"gene": n, "pagerank": round(pr[n], 4), "gwas_p": d.get("gwas_p"),
         "druggability": d.get("druggability")}
        for n, d in G.nodes(data=True) if d["kind"] == "gene"]
biomarkers = pd.DataFrame(rows).sort_values("pagerank", ascending=False)
print("Candidate biomarkers for", DISEASE, "(PageRank + evidence):")
print(biomarkers.to_string(index=False))


## 6 · Communities

Provided. Modularity clustering groups densely connected nodes.

> **TODO (report):** pick ONE community and read it as a subtype or mechanism
> hypothesis.

In [ ]:
from networkx.algorithms.community import greedy_modularity_communities
comms = greedy_modularity_communities(G.to_undirected())
print(f"{len(comms)} communities:")
for i, c in enumerate(comms):
    print(f"  {i}: {sorted(c)}")


## 7 · Visualize and export

Provided. Draws your graph coloured by kind and exports your node/edge table.
Both the figure and the CSV are graded artifacts — commit them.

In [ ]:
import matplotlib.pyplot as plt

KIND_COLOR = {"gene":"#D41B2C","disease":"#3E6F99","phenotype":"#2FA3D6",
              "pathway":"#00A58C","drug":"#9A5FC0","variant":"#FF854F","patient":"#8A94A3"}
H = G.subgraph([n for n, d in G.nodes(data=True) if d["kind"] != "patient"])
pos = nx.spring_layout(H, seed=7, k=0.7)
plt.figure(figsize=(13, 8))
nx.draw_networkx_edges(H, pos, edge_color="#C2CCD6", width=1.0, arrows=False)
nx.draw_networkx_nodes(H, pos, node_size=650,
                       node_color=[KIND_COLOR[H.nodes[n]["kind"]] for n in H.nodes()])
nx.draw_networkx_labels(H, pos, font_size=7)
plt.axis("off"); plt.title(f"{DISEASE} knowledge graph"); plt.tight_layout()
plt.savefig("my_knowledge_graph.png", dpi=150, bbox_inches="tight")   # <- commit this
plt.show()

# export node and edge tables (commit the CSVs)
pd.DataFrame([dict(name=n, **d) for n, d in NODES.items()]).to_csv("nodes.csv", index=False)
pd.DataFrame(EDGES).to_csv("edges.csv", index=False)
print("saved my_knowledge_graph.png, nodes.csv, edges.csv")

## 8 · Your report (the 40%)

Write a **3–4 page report** answering these four, in your own words:

1. **Schema and evidence** — why these node and edge types, and what evidence you put on the edges.
2. **Top targets** — which genes rank highest by centrality (§5), and does that match the biology you expect? Any non-obvious hit?
3. **A hypothesis from a path** — one repurposing or shared-mechanism hypothesis a path in your graph suggests (§4), and what would test it.
4. **Limitations** — an edge is evidence, not truth. Which of your edges are weak, and how would the answer change with better data? Note where you used (or would use) AI to harmonize data or write queries, and how you checked it.

#### Submit
- This notebook (**Kernel → Restart & Run All** first; credentials in a git-ignored file, not committed).
- `my_knowledge_graph.png`, `nodes.csv`, `edges.csv`.
- Your 3–4 page report (PDF) and a short README (name + your disease).
- Push to your biot6900 repo and post the link on Canvas.

#### Reset (to start over)
```cypher
MATCH (n) DETACH DELETE n      // wipes the database — run deliberately
```
